# 01 PuLPで学ぶ数理最適化の入門：どの活動を選ぶか

「テーマパークでの一日の回り方を提案したい」という目標に向けた、最初の一歩です。
このNotebookでは、**どの活動を選ぶか**だけを決める小さな問題を、PuLP（数理最適化のライブラリ）で解きます。

| 学ぶこと | 内容 |
| --- | --- |
| 何を入力するか | 時間の上限、予算、各活動の所要時間・希望度・費用、必須の活動 |
| 何を決めるか | 各活動を「選ぶ（1）」か「選ばない（0）」か |
| 何を守るか | 時間の合計 ≤ 上限、費用の合計 ≤ 予算、必須の活動は選ぶ |
| 何を最大化するか | 選んだ活動の希望度の合計 |

**最初に行う操作**: メニューの「ランタイム → すべてのセルを実行」を選ぶか、上のセルから順に `Shift + Enter` で実行します。
GPUは不要です（CPUのランタイムで動きます）。

- 活動・時間・費用・希望度は、すべて**計算例のための架空の値**です。費用は一般的な架空の追加料金の例で、実在のテーマパーク（ディズニーを含む）の施設ごとの料金を再現したものではありません。
- この入門では、**移動時間、時間帯ごとの混雑、ショーの開演時刻を省略**します。それらを含めて訪問順と時刻まで決めるのは、次の `02_day_plan_optimisation.ipynb` です（01とは別の問題です）。

## 0. セットアップ

次のセルは、(1) GitHub からこのリポジトリを HTTPS で取得し、(2) この作品（`projects/park-itinerary-lab`）を入門用の依存（PuLP と CBC ソルバー）付きでインストールし、(3) デモ設定を読み込みます。

- `REPO_REF` は取得するコードの版（ブランチ名・タグ名・コミットSHA）です。開発中は作業ブランチを指定しています。main に反映された後は `"main"` に変えます。**Colab で PR の Notebook を開いても、セットアップが自動的に PR のコードを取得するわけではありません**。必ず `REPO_REF` を確認してください。
- このリポジトリをローカルに取得して Jupyter で開いた場合は、`REPO_REF` を使わず、そのチェックアウトのコードを使います。
- `REPO_REF` を変えて再実行し「ランタイムを再起動」と表示されたら、メニューの「ランタイム → セッションを再起動する」を選び、このセルから実行し直します（読み込み済みの古いコードと混ざらないようにするためです）。
- 2回目以降の実行では、前回取得したコミットをそのまま使います（最新にするには `UPDATE_CHECKOUT = True`）。
- 取得とインストールの後は、計算にインターネット接続は使いません。Colab の `/content/` に置いたファイルは、ランタイムが終わると消えます。

In [ ]:
# @title セットアップ（最初に1回実行します。中身は読まなくても使えます）
# リポジトリの取得 → 作品のインストール → デモ設定の読み込み を行います。
REPO_URL = "https://github.com/shogo-ishikawa/2026_3Q_M4Th4.git"
REPO_REF = "claude/new-session-cc8kr0"  # @param {type:"string"}
CHECKOUT_DIR = "/content/2026_3Q_M4Th4"  # Colab で取得する場所（Colab 以外では ./2026_3Q_M4Th4）
UPDATE_CHECKOUT = False                  # True にすると、同じ REPO_REF でももう一度取得して最新にする
PROJECT_DIR_OVERRIDE = ""                # 配布zipを展開したフォルダを使う場合に指定（例: "/content/v0.1.0"）
PROJECT_SUBDIR = "projects/park-itinerary-lab"
INSTALL_EXTRA = "intro"                  # この Notebook に必要な追加の依存（01: intro、02: main）
NOTEBOOK_VERSION = "0.1.0"               # この Notebook が対応する作品の版

import importlib
import importlib.util
import json
import os
import platform
import re
import shutil
import subprocess
import sys
import tempfile
from importlib import metadata
from pathlib import Path

if sys.version_info < (3, 12):
    raise RuntimeError(f"Python 3.12 以上が必要です（PuLP 4 の要件）。今の版: {platform.python_version()}")

GIT_ENV = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}  # 認証の入力待ちで止まらないようにする
VERIFIED_VERSIONS = {"pulp": "4.2.0", "cbcbox": "2.929", "ortools": "9.15.6755"}
WATCHED_MODULES = {"numpy": "numpy", "pandas": "pandas", "matplotlib": "matplotlib", "google.protobuf": "protobuf"}


def run_git(args, cwd):
    """git コマンドを実行する。失敗したら、理由を表示して止める。"""
    done = subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True, env=GIT_ENV, timeout=300)
    if done.returncode != 0:
        raise RuntimeError(f"git {' '.join(args)} が失敗しました:\n{done.stderr.strip()}")
    return done.stdout.strip()


def git_or_none(args, cwd):
    try:
        return run_git(args, cwd)
    except (RuntimeError, OSError, subprocess.TimeoutExpired):
        return None


def is_project_dir(path):
    pyproject = Path(path) / "pyproject.toml"
    return pyproject.is_file() and 'name = "park-itinerary-lab"' in pyproject.read_text(encoding="utf-8")


def find_local_project(start):
    """今いるフォルダとその親から、この作品のフォルダを探す（repoルート・作品フォルダ・notebooks のどこからでも）。"""
    for folder in [start, *start.parents]:
        for candidate in (folder, folder / PROJECT_SUBDIR):
            if is_project_dir(candidate):
                return candidate.resolve()
    return None


def same_url(a, b):
    def clean(url):
        return url.strip().rstrip("/").removesuffix(".git").lower()
    return clean(a) == clean(b)


def fetch_commit(repo_dir, ref):
    """REPO_REF（ブランチ・タグ・コミットSHA・refs/pull/番号/head）を取得し、コミットSHAを返す。"""
    try:
        run_git(["fetch", "--quiet", "origin", ref], repo_dir)
        return run_git(["rev-parse", "FETCH_HEAD^{commit}"], repo_dir)
    except (RuntimeError, subprocess.TimeoutExpired) as error:
        first_error = error
    if re.fullmatch(r"[0-9a-fA-F]{7,40}", ref):  # 短いSHAは、すべてのブランチとタグを取得してから探す
        run_git(["fetch", "--quiet", "--tags", "origin", "+refs/heads/*:refs/remotes/origin/*"], repo_dir)
        found = git_or_none(["rev-parse", "--verify", "--quiet", f"{ref}^{{commit}}"], repo_dir)
        if found:
            return found
    raise RuntimeError(
        f"REPO_REF = {ref!r} を取得できませんでした（main には切り替えません）。"
        f"ブランチ名・タグ名・SHA とネットワークを確認してください。\n{first_error}"
    )


def prepare_clone(checkout_dir):
    """CHECKOUT_DIR に REPO_REF のコードを用意する。別のリポジトリや保存されていない変更は上書きしない。"""
    if not checkout_dir.exists():
        if git_or_none(["rev-parse", "--show-toplevel"], checkout_dir.parent):
            raise RuntimeError(f"{checkout_dir.parent} は別のGitリポジトリの中です。CHECKOUT_DIR を変えてください。")
        checkout_dir.mkdir(parents=True)
        run_git(["init", "--quiet"], checkout_dir)
        run_git(["remote", "add", "origin", REPO_URL], checkout_dir)
        run_git(["config", "--local", "parklab.managed", "true"], checkout_dir)
    if git_or_none(["config", "--local", "--get", "parklab.managed"], checkout_dir) != "true":
        raise RuntimeError(
            f"{checkout_dir} は、このセルが作ったチェックアウトではないため変更しません。"
            "CHECKOUT_DIR を変えるか、そのチェックアウトの中で Notebook を開いてください。"
        )
    origin = run_git(["remote", "get-url", "origin"], checkout_dir)
    if not same_url(origin, REPO_URL):
        raise RuntimeError(f"{checkout_dir} は別のリポジトリ（{origin}）です。CHECKOUT_DIR を変えてください。")
    head = git_or_none(["rev-parse", "--verify", "--quiet", "HEAD"], checkout_dir)
    last_ref = git_or_none(["config", "--local", "--get", "parklab.ref"], checkout_dir)
    last_commit = git_or_none(["config", "--local", "--get", "parklab.commit"], checkout_dir)
    if not UPDATE_CHECKOUT and head and last_ref == REPO_REF and head == last_commit:
        print(f"前回取得したコミット {head} を使います（最新にするには UPDATE_CHECKOUT = True）。")
        return checkout_dir / PROJECT_SUBDIR, head
    commit = fetch_commit(checkout_dir, REPO_REF)
    if git_or_none(["cat-file", "-e", f"{commit}:{PROJECT_SUBDIR}/pyproject.toml"], checkout_dir) is None:
        raise RuntimeError(
            f"REPO_REF = {REPO_REF!r}（コミット {commit}）には {PROJECT_SUBDIR} がありません。"
            "まだその版に反映されていない可能性があります。README で REPO_REF を確認してください。"
        )
    changes = run_git(["status", "--porcelain"], checkout_dir)
    if head != commit:
        if changes:
            raise RuntimeError(f"{checkout_dir} に保存されていない変更があるため、{REPO_REF} へ切り替えません。\n{changes}")
        run_git(["checkout", "--quiet", "--detach", commit], checkout_dir)
        print(f"チェックアウトを {head or '（なし）'} から {commit} に切り替えました。")
    elif changes:
        print("注意: チェックアウトに保存されていない変更があります（そのまま使います）:\n" + changes)
    run_git(["config", "--local", "parklab.ref", REPO_REF], checkout_dir)
    run_git(["config", "--local", "parklab.commit", commit], checkout_dir)
    return checkout_dir / PROJECT_SUBDIR, commit


def same_sources(installed_dir, source_dir):
    """インストールされたパッケージのファイルが、取得したコードと同じかを調べる。"""
    names = sorted(path.name for path in Path(source_dir).glob("*.py"))
    if names != sorted(path.name for path in Path(installed_dir).glob("*.py")):
        return False
    return all((Path(installed_dir) / n).read_bytes() == (Path(source_dir) / n).read_bytes() for n in names)


def installed_matches(source_dir):
    spec = importlib.util.find_spec("park_itinerary_lab")
    if spec is None or not spec.submodule_search_locations:
        return False
    return same_sources(list(spec.submodule_search_locations)[0], source_dir)


def install_project(project_dir):
    """作品を一時フォルダにコピーしてからインストールする（チェックアウトの中に build/ などを作らない）。"""
    with tempfile.TemporaryDirectory() as temp:
        copy = Path(temp) / "park-itinerary-lab"
        copy.mkdir()
        for name in ("pyproject.toml", "README.md"):
            shutil.copy2(project_dir / name, copy / name)
        shutil.copytree(project_dir / "src", copy / "src", ignore=shutil.ignore_patterns("__pycache__", "*.egg-info", "build"))
        done = subprocess.run(
            [sys.executable, "-m", "pip", "install", "--quiet", f"{copy}[{INSTALL_EXTRA}]"],
            capture_output=True, text=True,
        )
    if done.returncode != 0:
        raise RuntimeError("pip install に失敗しました:\n" + done.stdout[-2000:] + done.stderr[-3000:])
    importlib.invalidate_caches()


# 1. コードの場所を決める（指定のフォルダ → ローカルのチェックアウト → REPO_REF の取得 の順）
managed_dir = Path(CHECKOUT_DIR) if Path("/content").is_dir() else Path.cwd() / "2026_3Q_M4Th4"
local_project = Path(PROJECT_DIR_OVERRIDE).resolve() if PROJECT_DIR_OVERRIDE else find_local_project(Path.cwd())
if local_project is not None and not local_project.is_relative_to(managed_dir.resolve()):
    if not is_project_dir(local_project):
        raise RuntimeError(f"{local_project} はこの作品のフォルダではありません。")
    PROJECT_DIR = local_project
    COMMIT = git_or_none(["rev-parse", "HEAD"], PROJECT_DIR)
    print("この環境にある作品フォルダを使います（REPO_REF は使いません）。")
    if COMMIT and run_git(["status", "--porcelain", "--", "."], PROJECT_DIR):
        print("注意: 作品フォルダに、コミットしていない変更があります。")
    if not COMMIT and (PROJECT_DIR / "BUILD_INFO.json").is_file():
        print("配布zipの情報:", (PROJECT_DIR / "BUILD_INFO.json").read_text(encoding="utf-8").strip())
else:
    PROJECT_DIR, COMMIT = prepare_clone(managed_dir)
    print(f"{REPO_URL} の {REPO_REF} を {managed_dir} に用意しました。")
SOURCE_DIR = PROJECT_DIR / "src" / "park_itinerary_lab"
OUTPUT_DIR = PROJECT_DIR / "outputs"
SOURCE_INFO = {"repo_url": REPO_URL, "repo_ref": REPO_REF, "commit": COMMIT, "project_dir": str(PROJECT_DIR)}

# 2. インストール（すでに同じコードが入っていて、必要なライブラリもあれば省略）
if "park_itinerary_lab" in sys.modules:
    loaded_dir = Path(sys.modules["park_itinerary_lab"].__file__).parent
    if not same_sources(loaded_dir, SOURCE_DIR):
        raise RuntimeError(
            "前に読み込んだ park_itinerary_lab が、今のコードと違います。"
            "メニューの「ランタイム → セッションを再起動する」の後、このセルからもう一度実行してください。"
        )
NEEDED = {"intro": ["pulp", "cbcbox"], "main": ["ortools", "matplotlib", "matplotlib_fontja"]}[INSTALL_EXTRA]
loaded_before = {module: getattr(sys.modules[module], "__version__", None) for module in WATCHED_MODULES if module in sys.modules}
if not (installed_matches(SOURCE_DIR) and all(importlib.util.find_spec(m) for m in NEEDED)):
    print("作品をインストールしています（1〜2分かかることがあります）…")
    install_project(PROJECT_DIR)
changed = [m for m, version in loaded_before.items() if version != metadata.version(WATCHED_MODULES[m])]
if changed:
    raise RuntimeError(f"読み込み済みの {changed} の版が変わりました。ランタイムを再起動してから、このセルをもう一度実行してください。")

# 3. 読み込んで、取得したコードと同じかを確かめる
import park_itinerary_lab

if not same_sources(Path(park_itinerary_lab.__file__).parent, SOURCE_DIR):
    raise RuntimeError("読み込んだパッケージが取得したコードと一致しません。ランタイムを再起動して、もう一度実行してください。")
if park_itinerary_lab.__version__ != NOTEBOOK_VERSION:
    raise RuntimeError(
        f"この Notebook は版 {NOTEBOOK_VERSION} 用ですが、取得したコードは版 {park_itinerary_lab.__version__} です。"
        "同じ版の Notebook を開くか、REPO_REF を確認してください。"
    )

print("作品フォルダ (PROJECT_DIR):", PROJECT_DIR)
print("コミット:", COMMIT or "Gitのチェックアウトではありません（配布zipなど）")
print("読み込んだモジュール:", park_itinerary_lab.__file__)
print("Python:", platform.python_version(), "／", platform.platform())
for name in ["park-itinerary-lab", "pandas", *{"intro": ["pulp", "cbcbox"], "main": ["ortools", "matplotlib"]}[INSTALL_EXTRA]]:
    version = metadata.version(name)
    note = "" if VERIFIED_VERSIONS.get(name, version) == version else f"（注意: 確認した版は {VERIFIED_VERSIONS[name]}）"
    print(f"{name}: {version}{note}")

# 4. デモ設定（架空の活動）を読み込む
INTRO_DEMO = json.loads((PROJECT_DIR / "configs" / "intro_activities.json").read_text(encoding="utf-8"))
print("デモ設定:", INTRO_DEMO["meta"]["title"], f"（活動 {len(INTRO_DEMO['activities'])} 件）")

## 1. 問題の設定：7つの架空の活動

所要時間（分）は、平均的な待ち時間も含めた**固定の値**とします。希望度は「どれくらいやりたいか」を表す整数で、大きいほどやりたい活動です。

In [ ]:
import pandas as pd

pd.DataFrame(INTRO_DEMO["activities"]).rename(
    columns={"id": "ID", "name": "活動", "minutes": "所要時間(分)", "preference": "希望度", "cost_yen": "費用(円, 1人分)"}
)

## 2. 設定セル（ここを書き換えて試します）

学生が変更する値は、このセルにまとめています。値を変えたら、**このセルから下を順に**実行し直してください。

In [ ]:
import copy

activities = copy.deepcopy(INTRO_DEMO["activities"])  # 上の表の7つの活動（デモ設定のコピー）
time_limit_min = 360                                  # 使える時間の合計（分）。360分 = 6時間
budget_yen = 2000                                     # 追加で使える費用の上限（円, 1人分）
required_ids = ["F"]                                  # 必ず選ぶ活動のID（F: パレード鑑賞）

# 書き換えの例:
# activities[0]["preference"] = 5     # ジェットコースターの希望度を 5 にする
# budget_yen = 0                       # 有料の活動を選べないようにする
print("活動の数:", len(activities), "／時間の上限:", time_limit_min, "分／予算:", budget_yen, "円／必須:", required_ids)

## 3. 数式で書くと

活動を $i$（A〜G）とし、所要時間を $t_i$、希望度を $p_i$、費用を $c_i$、時間の上限を $T$、予算を $B$ とします。

- **決定変数**: $x_i \in \{0, 1\}$（活動 $i$ を選ぶなら 1、選ばないなら 0）
- **目的関数**: $\displaystyle \max \sum_i p_i x_i$（選んだ活動の希望度の合計を最大にする）
- **制約条件**:
  - 時間: $\displaystyle \sum_i t_i x_i \le T$
  - 予算: $\displaystyle \sum_i c_i x_i \le B$
  - 必須: 必須の活動 $m$ について $x_m = 1$

この形の問題は「0/1ナップサック問題」に必須の条件を加えたものです。次から、数式の各行を PuLP のコードに対応させて書きます。

### 3.1 問題を作る

`LpMaximize` は「最大化する問題」という意味です。

In [ ]:
import pulp

problem = pulp.LpProblem("activity_selection", pulp.LpMaximize)
print("問題の名前:", problem.name, "／向き: 最大化（変数・目的・制約はまだ空です）")

### 3.2 決定変数 $x_i$ を作る

PuLP 4 では、変数を `problem.add_variable(名前, cat="Binary")` で作ります（`Binary` は 0 か 1 だけをとる変数）。
`choose["A"]` が $x_A$ に当たります。

In [ ]:
choose = {}  # 活動ID → 決定変数 を入れる辞書
for activity in activities:
    choose[activity["id"]] = problem.add_variable(f"choose_{activity['id']}", cat="Binary")

# 内包表記を使うと、上の3行は次の1行と同じ意味です:
# choose = {a["id"]: problem.add_variable(f"choose_{a['id']}", cat="Binary") for a in activities}
choose

### 3.3 目的関数 $\sum_i p_i x_i$

`problem += 式, "名前"` で、目的関数（最初に加えた式）や制約条件を問題に加えます。

In [ ]:
total_preference = 0
for activity in activities:
    total_preference += activity["preference"] * choose[activity["id"]]
problem += total_preference, "total_preference"

# pulp.lpSum と内包表記を使うと、次の1行と同じ意味です:
# problem += pulp.lpSum(a["preference"] * choose[a["id"]] for a in activities), "total_preference"
print(total_preference)

### 3.4 制約条件（時間・予算・必須）

In [ ]:
total_minutes = 0
total_cost = 0
for activity in activities:
    total_minutes += activity["minutes"] * choose[activity["id"]]
    total_cost += activity["cost_yen"] * choose[activity["id"]]

problem += total_minutes <= time_limit_min, "time_limit"   # 時間の合計 ≤ 上限
problem += total_cost <= budget_yen, "budget"              # 費用の合計 ≤ 予算
for activity_id in required_ids:
    problem += choose[activity_id] == 1, f"required_{activity_id}"  # 必須の活動は選ぶ

for constraint in problem.constraints():
    print(constraint)

### 3.5 ソルバーで解く

PuLP は問題を「数式の形」にまとめる道具で、実際に解くのは**ソルバー**です。ここでは CBC（COIN-OR のソルバー）を使います。
CBC は、0/1 の組合せを1つずつすべて試すのではなく、「緩和した問題で上限を見積もり、見込みのない組合せをまとめて捨てる」分枝限定法などで、**最適であることを保証した解**を求めます。

PuLP 4 では CBC が PuLP 本体に含まれないため、`pulp[cbc]`（cbcbox）として導入しています。

**古い例や生成AIの提案に注意**: PuLP 4（このNotebookでは 4.2.0）では、古い版の書き方の一部が使えません。次のエラーが出たら、右の書き方に直してください。

| 古い書き方 | PuLP 4.2.0 で出るエラー | PuLP 4 での書き方 |
| --- | --- | --- |
| `pulp.LpVariable("x", cat="Binary")` | `TypeError: LpVariable.__init__() got an unexpected keyword argument 'cat'` | `problem.add_variable("x", cat="Binary")` |
| `pulp.LpStatus[problem.status]` | `AttributeError: module 'pulp' has no attribute 'LpStatus'` | `stats = problem.solve(...)` の後に `stats.status_str` |
| `pulp.PULP_CBC_CMD(msg=False)` | `AttributeError: module 'pulp' has no attribute 'PULP_CBC_CMD'` | `pulp.COIN_CMD(msg=False)`（CBC は `pulp[cbc]` で導入） |

In [ ]:
solver = pulp.COIN_CMD(msg=False)   # msg=False: ソルバーの途中経過を表示しない
stats = problem.solve(solver)
print("終了状態:", stats.status_str)          # Optimal なら最適解が見つかった
print("目的関数の値（希望度の合計）:", pulp.value(problem.objective))
print("ソルバー:", solver.name, "／CBC の場所:", solver.path)

In [ ]:
rows = []
for activity in activities:
    selected = pulp.value(choose[activity["id"]]) > 0.5
    rows.append({"ID": activity["id"], "活動": activity["name"], "選ぶ": "○" if selected else "",
                 "所要時間(分)": activity["minutes"], "費用(円)": activity["cost_yen"], "希望度": activity["preference"]})
inline_table = pd.DataFrame(rows)
inline_table

## 4. 作品のコード（src/）の関数で同じ問題を解く

上のセルは学習用に1行ずつ書いた例です。作品では、同じモデルを `src/park_itinerary_lab/intro.py` の `solve_selection()` にまとめています（演習ではこの関数を使います）。
両者の答えが一致することを確かめます。

In [ ]:
from park_itinerary_lab.intro import compare_with_enumeration, enumerate_selections, solve_selection

result = solve_selection(activities, time_limit_min, budget_yen, required_ids)
print(result)
assert result.status == stats.status_str
if result.status == "Optimal":
    assert result.total_preference == round(pulp.value(problem.objective))
    print("1行ずつ書いたモデルと、src/ の関数の答え（希望度の合計）は一致しました。")

## 5. 全列挙で答えを確かめる（$2^n$ 通り）

活動が7つなら、選び方は $2^7 = 128$ 通りしかありません。すべての選び方を調べ（全列挙）、条件を満たすものの中で希望度の合計が最大の値を求めて、ソルバーの答えと照合します。
同じ最大値をとる選び方が複数ある場合（同点）、ソルバーがそのどれを返しても正解なので、**照合するのは最大値と実行可能性**です。

全列挙は活動が増えると急に遅くなります（30個なら約10億通り）。ソルバーが必要になる理由です。

In [ ]:
enumeration = enumerate_selections(activities, time_limit_min, budget_yen, required_ids)
check = compare_with_enumeration(result, enumeration)
print("調べた選び方:", enumeration["n_total"], "通り／条件を満たす選び方:", enumeration["n_feasible"], "通り")
print("全列挙の最大値:", enumeration["best_value"], "／最大値をとる選び方:", enumeration["best_sets"])
print("照合結果:", check)
assert check["ok"], "ソルバーと全列挙の結果が一致しません"

In [ ]:
# 条件を満たす選び方を、希望度の合計の大きい順に5つ表示する
feasible = [row for row in enumeration["rows"] if row["feasible"]]
feasible.sort(key=lambda row: row["total_preference"], reverse=True)
pd.DataFrame(feasible[:5]).rename(columns={
    "selected": "選んだ活動", "total_preference": "希望度の合計", "total_minutes": "時間の合計(分)",
    "total_cost": "費用の合計(円)", "feasible": "条件を満たす",
})

## 6. 注意：時間の合計が上限以下でも、一日の予定が成立するとは限らない

この入門モデルは「所要時間の合計」だけを見ています。実際の一日では、

- 施設の間を**移動する時間**がかかります。
- 待ち時間は**並び始める時刻によって変わります**（昼は長く、夕方は短い等）。
- ショーには**開演時刻**があり、その前に着いている必要があります。

そのため、ここで選んだ活動を、実際に時刻表どおりに回れるとは限りません。どの順に回り、いつ並び始め、ショーのどの回を見るかまで同時に決めるのが、`02_day_plan_optimisation.ipynb` の本体モデル（OR-Tools CP-SAT）です。
01と02は別の問題ですが、「入力・決定変数・制約条件・目的関数」で考える点は共通です。

## 7. 演習

`solve_selection()` と `enumerate_selections()` を使って、次の問いに答えてみましょう。解答例は、それぞれ下のセルにあります（先に自分で予想してから実行してみましょう）。

1. 予算を **0円** にすると、選ばれる活動と希望度の合計はどう変わりますか。
2. 「レストランで昼食（G）」も**必須**にすると、何が選ばれなくなりますか。それはなぜですか。
3. 時間の上限を **120分** にし、必須を **A と F** にすると、どうなりますか。

### 演習1の解答例

In [ ]:
answer1 = solve_selection(activities, time_limit_min, 0, required_ids)
print(answer1)
print(compare_with_enumeration(answer1, enumerate_selections(activities, time_limit_min, 0, required_ids)))
# 費用のかかる D・E・G を選べなくなり、費用0円の活動だけから選びます。

### 演習2の解答例

In [ ]:
answer2 = solve_selection(activities, time_limit_min, budget_yen, ["F", "G"])
print(answer2)
print(compare_with_enumeration(answer2, enumerate_selections(activities, time_limit_min, budget_yen, ["F", "G"])))
# 昼食（1000円）を必ず入れると、残りの予算が1000円になり、1200円の体験工房（E）を選べなくなります。

### 演習3の解答例

In [ ]:
answer3 = solve_selection(activities, 120, budget_yen, ["A", "F"])
check3 = compare_with_enumeration(answer3, enumerate_selections(activities, 120, budget_yen, ["A", "F"]))
print(answer3.status, check3)
# A（90分）と F（60分）だけで150分になり、120分に収まりません。
# ソルバーは「Infeasible（実行不可能）」を返し、全列挙でも条件を満たす選び方は0通りです。
# 必須の条件を勝手に外して答えを作ることはしません。

## まとめ

- 数理最適化では「入力」「決定変数」「制約条件」「目的関数」を決めて、ソルバーに解いてもらいます。
- 小さな問題では全列挙と照合して、答えが正しいことを確かめられます。
- 移動・時間帯で変わる待ち時間・ショーの時刻を扱うには、訪問順と時刻も決めるモデルが必要です → `02_day_plan_optimisation.ipynb` へ進みましょう。